# Pipeline Combined (Preprocess + Modeling)

End-to-End Pipeline without leakage, including `score_persist7` feature.

# Preprocessing (v2)

Consolidated monolithic version.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# --- Environment Detection ---
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

# --- Setup Paths ---
if IS_KAGGLE:
    print("Umgebung: Kaggle")
    DATA_DIR = Path("/kaggle/input/datasets/axxtur/data-mining-2026-final-assignment")
    OUTPUTS_DIR = Path("/kaggle/working/outputs")
else:
    print("Umgebung: Lokal")
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    root = Path.cwd()
    for c in candidates:
        if (c / "data" / "test.csv").exists():
            root = c
            break
    DATA_DIR = root / "data"
    OUTPUTS_DIR = root / "outputs"

if IS_KAGGLE and not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/data-mining-final-project")

TRAIN_PATH = DATA_DIR / "train.csv"
MODE = "full"
if not TRAIN_PATH.exists():
    TRAIN_PATH = DATA_DIR / "train_sample.csv"
    MODE = "sample"

TEST_PATH = DATA_DIR / "test.csv"
PROCESSED_DIR = OUTPUTS_DIR / "processed"
SUBMISSION_DIR = OUTPUTS_DIR / "submissions"
FIGURES_DIR = OUTPUTS_DIR / "figures"

for d in [PROCESSED_DIR, SUBMISSION_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

env = {
        "is_kaggle": IS_KAGGLE,
    "mode": MODE,
    "data_dir": DATA_DIR,
    "train_path": TRAIN_PATH,
    "test_path": TEST_PATH,
    "processed_dir": PROCESSED_DIR,
    "submission_dir": SUBMISSION_DIR,
    "outputs_dir": OUTPUTS_DIR,
    "figures_dir": FIGURES_DIR,
}

print(f"Modus: {MODE}")
print(f"Train Path: {TRAIN_PATH} ({'OK' if TRAIN_PATH.exists() else 'FEHLT'})")
print(f"Test Path:  {TEST_PATH} ({'OK' if TEST_PATH.exists() else 'FEHLT'})")


## Script: features_v2.py

In [ ]:
def add_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby("region_id", sort=False)
    
    # 1. Standard rolling means/max
    for col in ROLL_COLS:
        prior = grouped[col].shift(1)
        for window in ROLL_WINDOWS:
            roller = prior.groupby(df["region_id"], sort=False)
            df[f"{col}_roll{window}_mean"] = roller.transform(lambda s: s.rolling(window, min_periods=3).mean())
            df[f"{col}_roll{window}_max"] = roller.transform(lambda s: s.rolling(window, min_periods=3).max())
            if col == "prec":
                df[f"prec_roll{window}_sum"] = roller.transform(lambda s: s.rolling(window, min_periods=3).sum())
    
    # 2. V4 & New Meteorological Features (Shifted to prevent leak)
    prior_prec = grouped["prec"].shift(1)
    prior_tmp = grouped["tmp"].shift(1)
    prior_hum = grouped["humidity"].shift(1)
    
    roller_prec = prior_prec.groupby(df["region_id"], sort=False)
    roller_tmp = prior_tmp.groupby(df["region_id"], sort=False)
    roller_hum = prior_hum.groupby(df["region_id"], sort=False)
    
    # Base anomalies
    df["prec_roll90_mean"] = roller_prec.transform(lambda s: s.rolling(90, min_periods=10).mean())
    df["prec_roll365_mean"] = roller_prec.transform(lambda s: s.rolling(365, min_periods=60).mean())
    df["tmp_roll90_mean"] = roller_tmp.transform(lambda s: s.rolling(90, min_periods=10).mean())
    df["tmp_roll365_mean"] = roller_tmp.transform(lambda s: s.rolling(365, min_periods=60).mean())
    
    # Drought Indices (V4)
    df["prec_deficit_90d"] = df["prec_roll90_mean"] - df["prec_roll365_mean"]
    df["tmp_anomaly_90d"] = df["tmp_roll90_mean"] - df["tmp_roll365_mean"]
    
    p7 = roller_prec.transform(lambda s: s.rolling(7, min_periods=3).mean())
    p30 = roller_prec.transform(lambda s: s.rolling(30, min_periods=10).mean())
    p30_std = roller_prec.transform(lambda s: s.rolling(30, min_periods=10).std().clip(lower=0.01))
    df["prec_trend_30d"] = (p7 - p30) / p30_std
    
    hum_90 = roller_hum.transform(lambda s: s.rolling(90, min_periods=30).mean())
    hum_365 = roller_hum.transform(lambda s: s.rolling(365, min_periods=60).mean())
    df["humidity_deficit_90d"] = hum_90 - hum_365
    
    df["heat_drought_idx"] = df["prec_deficit_90d"] * df["tmp_anomaly_90d"].clip(lower=0)
    
    # Dry days counts
    is_dry = (prior_prec < 1.0).astype(float)
    dry_roller = is_dry.groupby(df["region_id"], sort=False)
    df["dry_days_14d"] = dry_roller.transform(lambda s: s.rolling(14, min_periods=3).sum())
    df["dry_days_30d"] = dry_roller.transform(lambda s: s.rolling(30, min_periods=7).sum())
    
    # NEW Domain Features: VPD (Vapor Pressure Deficit)
    # SVP = 0.6108 * exp(17.27 * T / (T + 237.3))
    # AVP = SVP * (RH / 100)
    # VPD = SVP - AVP
    svp = 0.6108 * np.exp((17.27 * prior_tmp) / (prior_tmp + 237.3))
    avp = svp * (prior_hum / 100.0)
    vpd = svp - avp
    vpd_roller = vpd.groupby(df["region_id"], sort=False)
    df["vpd_mean_14d"] = vpd_roller.transform(lambda s: s.rolling(14, min_periods=3).mean())
    df["vpd_mean_30d"] = vpd_roller.transform(lambda s: s.rolling(30, min_periods=7).mean())
    
    # NEW Domain Features: DTR (Diurnal Temp Range)
    prior_dtr = grouped["tmp_range"].shift(1)
    dtr_roller = prior_dtr.groupby(df["region_id"], sort=False)
    df["dtr_mean_14d"] = dtr_roller.transform(lambda s: s.rolling(14, min_periods=3).mean())
    df["dtr_mean_30d"] = dtr_roller.transform(lambda s: s.rolling(30, min_periods=7).mean())
    
    # NEW Domain Features: EMA Prec
    df["prec_ema_14d"] = roller_prec.transform(lambda s: s.ewm(span=14, min_periods=3).mean())
    
    # NEW Domain Features: AET Proxy (Apparent Evapotranspiration)
    prior_wind = grouped["wind"].shift(1)
    aet = (prior_tmp * prior_wind) / (prior_hum + 1.0)
    aet_roller = aet.groupby(df["region_id"], sort=False)
    df["aet_proxy_30d"] = aet_roller.transform(lambda s: s.rolling(30, min_periods=7).mean())
    
    return df

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = parse_dates(df)
    df = add_ordinal(df)
    df = sort_panel(df)
    df = add_calendar_features(df)
    df = add_lag_features(df)
    df = add_rolling_features(df)
    return df

def feature_columns(include_region: bool = True) -> list[str]:
    lag_names = [f"{c}_lag{lag}" for c in LAG_COLS for lag in LAGS]
    roll_names = []
    for col in ROLL_COLS:
        for window in ROLL_WINDOWS:
            roll_names.extend([f"{col}_roll{window}_mean", f"{col}_roll{window}_max"])
            if col == "prec":
                roll_names.append(f"prec_roll{window}_sum")
                
    drought_indices = [
        "prec_deficit_90d", "tmp_anomaly_90d", "prec_trend_30d",
        "humidity_deficit_90d", "heat_drought_idx",
        "dry_days_14d", "dry_days_30d",
        "vpd_mean_14d", "vpd_mean_30d",
        "dtr_mean_14d", "dtr_mean_30d",
        "prec_ema_14d", "aet_proxy_30d"
    ]
    
    calendar = ["month_sin", "month_cos", "day_sin", "day_cos"]
    cols = list(WEATHER_COLS) + lag_names + roll_names + drought_indices + calendar
    if include_region:
        cols = ["region_id"] + cols
    return cols


## Script: parallel_util.py

In [ ]:
"""Shared parallelism helpers for preprocessing and modeling."""
from __future__ import annotations

import os
from concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait
from typing import Callable, Iterator, TypeVar

T = TypeVar("T")
R = TypeVar("R")


def default_workers(cap: int = 8) -> int:
    """
    Worker count: env ``DM_WORKERS``, else ``CPU-1`` (capped).

    Set ``DM_WORKERS=1`` to disable multiprocessing.
    """
    env = os.environ.get("DM_WORKERS", "").strip()
    if env:
        return max(1, int(env))
    import multiprocessing as mp

    return max(1, min(cap, mp.cpu_count() - 1))


def run_parallel_map(
    fn: Callable[[T], R],
    tasks: list[T],
    *,
    n_workers: int | None = None,
) -> list[R]:
    """``pool.map`` — order preserved; fine for small task lists (e.g. 5 weeks)."""
    if not tasks:
        return []
    n_workers = n_workers or default_workers()
    if n_workers <= 1:
        return [fn(t) for t in tasks]
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        return list(pool.map(fn, tasks))


def run_parallel_consume(
    fn: Callable[[T], R],
    tasks: Iterator[T],
    consume: Callable[[R], None],
    *,
    n_workers: int | None = None,
    max_inflight: int | None = None,
) -> int:
    """
    Stream tasks through a process pool; call *consume* on each result as ready.
    Returns number of tasks completed.
    """
    n_workers = n_workers or default_workers()
    max_inflight = max_inflight or n_workers * 2
    n_done = 0

    if n_workers <= 1:
        for task in tasks:
            consume(fn(task))
            n_done += 1
        return n_done

    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        inflight: dict = {}
        task_iter = iter(tasks)

        def submit_one() -> bool:
            try:
                task = next(task_iter)
            except StopIteration:
                return False
            inflight[pool.submit(fn, task)] = True
            return True

        for _ in range(max_inflight):
            if not submit_one():
                break

        while inflight:
            done, _ = wait(inflight, return_when=FIRST_COMPLETED)
            for fut in done:
                inflight.pop(fut)
                consume(fut.result())
                n_done += 1
                submit_one()
    return n_done


## Script: preprocess_streaming_v2.py

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

class _ParquetAppender:
    def __init__(self, path) -> None:
        self.path = path
        self._writer = None

    def write(self, df) -> None:
        if df.empty:
            return
        table = pa.Table.from_pandas(df, preserve_index=False)
        if self._writer is None:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            self._writer = pq.ParquetWriter(self.path, table.schema)
        self._writer.write_table(table)

    def close(self) -> None:
        if self._writer is not None:
            self._writer.close()
            self._writer = None

"""
Preprocessing v2 — streaming + optional multi-core per region.

Outputs:
  train_labeled_v2.parquet
  test_features_v2.parquet
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq





OUT_TRAIN_V2 = "train_labeled_v2.parquet"
OUT_TEST_V2 = "test_features_v2.parquet"


def process_region_v2_core(
    train_part: pd.DataFrame,
    test_part: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Feature engineering for one region (no I/O)."""
    train_part = train_part.copy()
    test_part = test_part.copy()
    train_part["_split"] = "train"
    test_part["_split"] = "test"
    if "score" not in test_part.columns:
        test_part["score"] = float("nan")

    panel = pd.concat([train_part, test_part], ignore_index=True)
    panel = build_features_v2(panel)

    train_feat = panel[panel["_split"] == "train"]
    test_feat = panel[panel["_split"] == "test"]
    test_raw = parse_dates(test_part.drop(columns=["_split"], errors="ignore"))

    train_labeled = train_feat[train_feat["score"].notna()]
    return (
        train_labeled[save_columns_v2(train_labeled, labeled=True)],
        test_feat[save_columns_v2(test_feat, labeled=False)],
    )


def _region_worker_v2(
    args: tuple[pd.DataFrame, pd.DataFrame],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_r, test_r = args
    return process_region_v2_core(train_r, test_r)


def _process_region_v2(
    train_part: pd.DataFrame,
    test_part: pd.DataFrame,
    train_writer: _ParquetAppender,
    test_writer: _ParquetAppender,
) -> None:
    train_out, test_out = process_region_v2_core(train_part, test_part)
    train_writer.write(train_out)
    test_writer.write(test_out)


def _iter_region_tasks(
    train_path: Path,
    test_by_region: dict,
    chunk_size: int,
):
    """Yield (train_r, test_r, region_stats) per completed region."""
    buffer_parts: list[pd.DataFrame] = []
    current_region = None

    for chunk in pd.read_csv(train_path, chunksize=chunk_size):
        chunk = parse_dates(chunk)
        for region, g in chunk.groupby("region_id", sort=False):
            if current_region is None:
                current_region = region
                buffer_parts = [g]
                continue

            if region == current_region:
                buffer_parts.append(g)
                continue

            train_r = pd.concat(buffer_parts, ignore_index=True)
            test_r = test_by_region.get(current_region, pd.DataFrame())
            yield (train_r, test_r)
            current_region = region
            buffer_parts = [g]

    if current_region is not None and buffer_parts:
        train_r = pd.concat(buffer_parts, ignore_index=True)
        test_r = test_by_region.get(current_region, pd.DataFrame())
        yield (train_r, test_r)


def preprocess_by_region_v2(
    train_path: Path,
    test_path: Path,
    out_train: Path,
    out_test: Path,
    chunk_size: int = 500_000,
    n_workers: int | None = None,
) -> dict:
    n_workers = n_workers if n_workers is not None else default_workers()
    
    

    test = parse_dates(pd.read_csv(test_path))
    test_by_region = {r: g for r, g in test.groupby("region_id", sort=False)}

    for path in (out_train, out_test):
        if path.exists():
            path.unlink()

    train_writer = _ParquetAppender(out_train)
    test_writer = _ParquetAppender(out_test)
    finished_regions: set[str] = set()
    duplicate_test_skipped = 0

    def _consume(result: tuple[pd.DataFrame, pd.DataFrame]) -> None:
        nonlocal duplicate_test_skipped
        train_out, test_out = result
        if train_out.empty and test_out.empty:
            return

        rid = str(
            test_out["region_id"].iloc[0]
            if not test_out.empty
            else train_out["region_id"].iloc[0]
        )

        if rid in finished_regions:
            duplicate_test_skipped += 1
            if not train_out.empty:
                train_writer.write(train_out)
            return

        finished_regions.add(rid)
        if not train_out.empty:
            train_writer.write(train_out)
        if not test_out.empty:
            test_writer.write(test_out)

        if len(finished_regions) % 200 == 0:
            print(f"  … {len(finished_regions)} Regionen verarbeitet")

    try:
        tasks = _iter_region_tasks(train_path, test_by_region, chunk_size)
        run_parallel_consume(
            _region_worker_v2,
            tasks,
            _consume,
            n_workers=n_workers,
        )
    finally:
        train_writer.close()
        test_writer.close()

    train_rows = pq.read_metadata(out_train).num_rows if out_train.exists() else 0
    test_rows = pq.read_metadata(out_test).num_rows if out_test.exists() else 0

    if duplicate_test_skipped:
        print(
            f"  Hinweis: {duplicate_test_skipped} doppelte Region-Durchläufe "
            "(test nur 1× geschrieben)."
        )

    return {
        "version": 2,
        "regions": len(finished_regions),
        "duplicate_region_passes": duplicate_test_skipped,
        "train_labeled_rows": train_rows,
        "test_rows": test_rows,
        "out_train": out_train,
        "out_test": out_test,
        "feature_count": len(feature_columns_v2()),
        "n_workers": n_workers,
    }


# 03b – Preprocessing v2 (erweiterte Features)

**Vergleich zu v1:** [docs/10_PREPROCESSING_V2.md](../docs/10_PREPROCESSING_V2.md) · Original: `03_preprocessing.ipynb`

| Output v2 | v1 (alt) |
|-----------|----------|
| `train_labeled_v2.parquet` | `train_labeled.parquet` |
| `test_features_v2.parquet` | `test_features.parquet` |

Neu: Score-Lags, `score_persist7` als Feature, Regional-Score-Stats, 91-Tage-Test-Aggregate.

**Parallel:** mehrere Regionen gleichzeitig (`DM_WORKERS`, Standard ≈ CPU−1). Kaggle-Tipp: CSV nach `/content/` kopieren (Zelle 2).

→ danach **`04b_modeling_v2.ipynb`**

In [ ]:
import os
import shutil


path_train_v2 = PROCESSED_DIR / "train_labeled_v2.parquet"
path_test_v2 = PROCESSED_DIR / "test_features_v2.parquet"
N_WORKERS = default_workers()

# Kaggle: /kaggle/input I/O vermeiden (oft schneller als Paid-CPU)
train_path = TRAIN_PATH
test_path = TEST_PATH
if env.get("is_kaggle"):
    local_train = Path("/kaggle/working/train.csv") if env.get("is_kaggle") else Path("/content/train.csv")
    local_test = Path("/kaggle/working/test.csv") if env.get("is_kaggle") else Path("/content/test.csv")
    if not local_train.exists():
        print("Kopiere train.csv → /content/ …")
        shutil.copy(TRAIN_PATH, local_train)
    if not local_test.exists():
        print("Kopiere test.csv → /content/ …")
        shutil.copy(TEST_PATH, local_test)
    train_path, test_path = local_train, local_test


if MODE == "full":
    print(f"Streaming v2 parallel ({N_WORKERS} workers) …")
    stats = preprocess_by_region_v2(
        train_path, test_path, path_train_v2, path_test_v2,
        chunk_size=500_000, n_workers=N_WORKERS,
    )
    print("Fertig:", stats)
else:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    train_labeled, test_feat = preprocess_panel_v2(train, test, n_workers=N_WORKERS)
    train_labeled.to_parquet(path_train_v2, index=False)
    test_feat.to_parquet(path_test_v2, index=False)
    print(f"Gespeichert: {len(train_labeled):,} train | {len(test_feat):,} test")


In [ ]:
import pyarrow.parquet as pq

n_train = pq.read_metadata(path_train_v2).num_rows
n_test = pq.read_metadata(path_test_v2).num_rows
print(f"Parquet v2: train {n_train:,} | test {n_test:,}")

head = pd.read_parquet(path_train_v2)
v2_only = [c for c in head.columns if c.startswith(("score_lag", "region_score", "test91_"))]
print(f"Neue Spalten (Beispiel): {v2_only[:8]} … ({len(v2_only)} v2-spezifisch sichtbar)")
head.head(3)


# Modeling — 5-Wochen-Vorhersage (v2)

Trainiert LightGBM auf den vom Preprocessing erzeugten Parquet-Dateien
und schreibt die Kaggle-Submission (`pred_week1..5`).

**Input** — `outputs/processed/`
- `train_labeled_v2.parquet` — wöchentlich gelabelte Zeilen inkl. Features
- `test_features_v2.parquet` — 91-Tage-Testfenster je Region inkl. Features

**Output** — `outputs/submissions/submission_<mode>_v2.csv`

**Pipeline:** Sliding-Window (Woche *i* → Score Woche *i+1..i+5*) ·
Region-Holdout · Baselines · 5 LightGBM-Modelle · Persistence-Blend.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

# --- Umgebung erkennen ---
IS_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") != ""

if IS_KAGGLE:
    OUTPUTS_DIR = Path("/kaggle/working/outputs")
else:
    root = next(
        (c for c in [Path.cwd(), *Path.cwd().parents] if (c / "outputs").is_dir()),
        Path.cwd(),
    )
    OUTPUTS_DIR = root / "outputs"

PROCESSED_DIR = OUTPUTS_DIR / "processed"
SUBMISSION_DIR = OUTPUTS_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PARQUET = PROCESSED_DIR / "train_labeled_v2.parquet"
TEST_PARQUET = PROCESSED_DIR / "test_features_v2.parquet"

# --- Konfiguration ---
RANDOM_STATE = 42
N_WEEKS = 5            # Vorhersagehorizont (pred_week1..5)
VAL_REGION_FRAC = 0.2  # Anteil Regionen fuer den Holdout
ES_ROUNDS = 50         # Early-Stopping-Geduld
BLEND_PERSIST = 0.35   # Gewicht der Persistence im finalen Blend

print(f"Umgebung: {'Kaggle' if IS_KAGGLE else 'Lokal'}")
print(f"Train: {TRAIN_PARQUET}  ({'OK' if TRAIN_PARQUET.exists() else 'FEHLT'})")
print(f"Test:  {TEST_PARQUET}  ({'OK' if TEST_PARQUET.exists() else 'FEHLT'})")


## 1 · Daten laden

In [ ]:
# --- Daten laden ---
MODE = "sample" if "sample" in TRAIN_PARQUET.name else "full"

print("Lade Parquet-Dateien...")
train_df = pd.read_parquet(TRAIN_PARQUET)
test_df = pd.read_parquet(TEST_PARQUET)

# Convert region_id to category for LightGBM
train_df["region_id"] = train_df["region_id"].astype("category")
test_df["region_id"] = test_df["region_id"].astype("category")

# Meta-Spalten zaehlen nicht als numerisches Modell-Feature.
# 'region_id' wurde wieder in META aufgenommen (wir nutzen stattdessen die region_stats)
META = {"date", "year", "month", "day", "ordinal", "score", "score_persist7"}
FEATURES = [c for c in train_df.columns if c not in META and c in test_df.columns]

# Cast only numeric features to float32 to save memory
num_features = [c for c in FEATURES if c != "region_id"]
train_df[num_features] = train_df[num_features].astype(np.float32)
test_df[num_features] = test_df[num_features].astype(np.float32)

print(f"Modus: {MODE}")
print(f"Train: {len(train_df):,} Zeilen · {train_df['region_id'].nunique():,} Regionen")
print(f"Test:  {len(test_df):,} Zeilen · {test_df['region_id'].nunique():,} Regionen")
print(f"Features ({len(FEATURES)}): {FEATURES[:6]} …")
train_df.head(3)


## 2 · Sliding-Window-Helfer

Die gelabelten Zeilen liegen pro Region bereits **woechentlich** vor — jede
Zeile ist eine Woche. Aus den Features der Woche *i* sagen wir die Scores der
Wochen *i+1 … i+5* voraus. Validiert wird auf **ungesehenen Regionen**
(je 1 Fenster, wie die Test-Situation).

In [ ]:
def clip_scores(arr):
    """Auf den Wettbewerbs-Score-Bereich begrenzen."""
    return np.clip(arr, 0.0, 5.0)


def mae(y_true, y_pred):
    """MAE ueber alle Regionen x 5 Wochen (Kaggle-Metrik)."""
    return float(np.mean(np.abs(clip_scores(y_pred) - np.asarray(y_true))))


def week_target(y, w):
    """Labels fuer Woche w als 1D-float64-Array (LightGBM 4.x erwartet 1D)."""
    return np.ascontiguousarray(y[:, w], dtype=np.float64)


def build_samples(weekly, mode="train", val_offset_weeks=0):
    """
    Sliding-Window je Region.
    val_offset_weeks: 0 = letztes Fenster, 5 = Fenster 5 Wochen davor, etc.
    mode='train': Alle Fenster VOR dem Validierungsfenster.
    mode='val': Genau das Validierungsfenster.
    mode='all': Alle verfuegbaren Fenster.
    """
    X_parts, y_parts, p_parts, r_parts = [], [], [], []
    for region, g in weekly.groupby("region_id", sort=True):
        g = g.sort_values("ordinal")
        scores = g["score"].to_numpy(np.float32)
        n = len(g)
        
        if n <= 2 * N_WEEKS + val_offset_weeks:
            continue
            
        y = np.lib.stride_tricks.sliding_window_view(scores[1:], N_WEEKS)
        X = g.iloc[: n - N_WEEKS][FEATURES]
        persist = scores[: n - N_WEEKS]
        
        target_val_idx = -1 - val_offset_weeks
        
        if mode == "val":
            # Das Validierungsfenster
            X, y, persist = X.iloc[[target_val_idx]], y[[target_val_idx]], persist[[target_val_idx]]
        elif mode == "train":
            # Alle Fenster davor (ohne Target-Ueberschneidung)
            # Das Fenster bei target_val_idx ueberschneidet sich die naechsten N_WEEKS.
            # Train darf hoechstens bis target_val_idx - N_WEEKS gehen.
            max_idx = len(X) + target_val_idx - N_WEEKS + 1
            if max_idx <= 0:
                continue
            X, y, persist = X.iloc[:max_idx], y[:max_idx], persist[:max_idx]
        elif mode == "all":
            pass # Behalte alles
            
        X_parts.append(X)
        y_parts.append(y)
        p_parts.append(persist)
        r_parts.append(np.full(len(persist), region, dtype=object))

    if not X_parts:
        return pd.DataFrame(), np.array([]), np.array([]), np.array([])
        
    X = pd.concat(X_parts, ignore_index=True)
    return X, np.vstack(y_parts), np.concatenate(p_parts), np.concatenate(r_parts)


## 3 · Train/Validierungs-Split

## 4 · Baselines (MAE wie Kaggle)

## 5 · LightGBM — 5 Wochen-Modelle

Ein Modell je Vorhersagewoche. Bewusst **konservative** Parameter (flache
Baeume, grosse Blaetter, L1/L2, Subsampling) gegen Overfitting auf den
~1,7 Mio. Fenstern. Early Stopping gegen den Region-Holdout.

In [ ]:
LGB_PARAMS = dict(
    objective="mae",
    n_estimators=1000,
    learning_rate=0.1,
    num_leaves=11,
    max_depth=5,
    min_child_samples=300,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.6,
    reg_alpha=3.0,
    reg_lambda=5.0,
    verbose=-1,
)

FOLDS = [0]
fold_maes = []
best_iters_per_week = {w: [] for w in range(N_WEEKS)}

for fold, offset in enumerate(FOLDS):
    print(f"\n{'='*40}\nFold {fold+1} (Offset {offset} Wochen)\n{'='*40}")
    
    X_tr, y_tr, _, _ = build_samples(train_df, mode="train", val_offset_weeks=offset)
    X_va, y_va, _, _ = build_samples(train_df, mode="val", val_offset_weeks=offset)
    
    if len(X_va) == 0:
        print("Nicht genug Daten für diesen Fold!")
        continue
        
    print(f"Train-Fenster: {len(X_tr):,} | Val-Fenster: {len(X_va):,}")
    
    val_preds = np.zeros_like(y_va, dtype=np.float64)
    for w in range(N_WEEKS):
        m = lgb.LGBMRegressor(**LGB_PARAMS, random_state=RANDOM_STATE + w + fold*100, n_jobs=-1)
        m.fit(
            X_tr, week_target(y_tr, w),
            eval_set=[(X_va, week_target(y_va, w))],
            eval_metric="mae",
            callbacks=[lgb.early_stopping(ES_ROUNDS, verbose=False)],
        )
        val_preds[:, w] = clip_scores(m.predict(X_va))
        best_iters_per_week[w].append(m.best_iteration_)
        
    lgb_mae = mae(y_va, val_preds)
    fold_maes.append(lgb_mae)
    
    print(f"LightGBM MAE: {lgb_mae:.4f}")

print(f"\n{'='*40}\nDURCHSCHNITT ÜBER ALLE FOLDS\n{'='*40}")
print(f"LightGBM MAE (nur Wetterdaten): {np.mean(fold_maes):.4f}")


## 6 · Finales Training & Submission

Neu-Training auf **allen** Regionen mit der je Woche validierten Baum-Anzahl
(`best_iteration`). Vorhersage je Region aus dem letzten Tag des
Testfensters, anschliessend Persistence-Blend.

In [ ]:
# Neu-Training auf ALLEN Regionen
X_all, y_all, _, _ = build_samples(train_df, mode="all")

final_models = []
for w in range(N_WEEKS):
    avg_best_iter = int(np.mean(best_iters_per_week[w])) if best_iters_per_week[w] else LGB_PARAMS["n_estimators"]
    
    m = lgb.LGBMRegressor(
        **{**LGB_PARAMS, "n_estimators": avg_best_iter},
        random_state=RANDOM_STATE + w,
        n_jobs=-1,
    )
    m.fit(X_all, week_target(y_all, w))
    final_models.append(m)
print(f"Finales Training: {N_WEEKS} Modelle auf {len(X_all):,} Fenstern")

# Eine Feature-Zeile je Region: letzter Tag des 91-Tage-Testfensters.
last_rows = test_df.sort_values(["region_id", "ordinal"]).groupby("region_id").tail(1)
test_regions = last_rows["region_id"].to_numpy()
X_test = last_rows[FEATURES]

test_preds = np.column_stack([clip_scores(m.predict(X_test)) for m in final_models])

# Keine Persistence mehr, reines Modell!
print("Submission: 100% LightGBM (Wetterdaten-Modell)")

submission = pd.DataFrame({"region_id": test_regions})
for k in range(N_WEEKS):
    submission[f"pred_week{k + 1}"] = test_preds[:, k]
submission = submission.sort_values("region_id").reset_index(drop=True)

out_path = SUBMISSION_DIR / f"submission_{MODE}_v2.csv"
submission.to_csv(out_path, index=False)
print(f"Gespeichert: {out_path}  ({len(submission):,} Zeilen)")
submission.head(10)
